### Data Invertory  資料盤點

In [119]:
import pandas as pd
from pathlib import Path 
import os
import numpy as np
pd.set_option("display.max_colwidth", None)      # 不截斷欄位內容
pd.set_option("display.max_rows", 200)           # 想看更多 row 可調
pd.set_option("display.max_columns", 50)   

In [64]:
PROJECT_ROOT = Path("/Users/kaiping/Desktop/olist_project")  # 定義專案根目錄(絕對路徑)
RAW = PROJECT_ROOT / "data" / "raw" # 組出raw資料夾路徑

df_translation = pd.read_csv(RAW / "product_category_name_translation.csv")
df_sellers     = pd.read_csv(RAW / "olist_sellers_dataset.csv")
df_products    = pd.read_csv(RAW / "olist_products_dataset.csv")
df_orders      = pd.read_csv(RAW / "olist_orders_dataset.csv")
df_order_reviews  = pd.read_csv(RAW / "olist_order_reviews_dataset.csv")
df_order_payments = pd.read_csv(RAW / "olist_order_payments_dataset.csv")
df_order_items    = pd.read_csv(RAW / "olist_order_items_dataset.csv")
df_geolocation    = pd.read_csv(RAW / "olist_geolocation_dataset.csv")
df_customers      = pd.read_csv(RAW / "olist_customers_dataset.csv")


In [65]:
df_translation = pd.read_csv(RAW / "product_category_name_translation.csv")
df_sellers     = pd.read_csv(RAW / "olist_sellers_dataset.csv")
df_products    = pd.read_csv(RAW / "olist_products_dataset.csv")
df_orders      = pd.read_csv(RAW / "olist_orders_dataset.csv")
df_order_reviews  = pd.read_csv(RAW / "olist_order_reviews_dataset.csv")
df_order_payments = pd.read_csv(RAW / "olist_order_payments_dataset.csv")
df_order_items    = pd.read_csv(RAW / "olist_order_items_dataset.csv")
df_geolocation    = pd.read_csv(RAW / "olist_geolocation_dataset.csv")
df_customers      = pd.read_csv(RAW / "olist_customers_dataset.csv")

In [66]:
tables = {
    "translation":df_translation,
    "sellers":df_sellers,
    "product":df_products,
    "orders":df_orders,
    "order_review":df_order_reviews,
    "order_payment":df_order_payments,
    "order_items":df_order_items,
    "geolocation":df_geolocation,
    "customer":df_customers}

In [67]:
inventory = []

for name, df in tables.items():
    inventory.append({
        "table_name": name,
        "row_count": df.shape[0], # 取第一維即是row數
        "column_count": df.shape[1] # 第二維
    })

df_inventory = pd.DataFrame(inventory)
df_inventory


,table_name,row_count,column_count
0,translation,71,2
1,sellers,3095,4
2,product,32951,9
3,orders,99441,8
4,order_review,99224,7
5,order_payment,103886,5
6,order_items,112650,7
7,geolocation,1000163,5
8,customer,99441,5


### Column inventory 欄位理解

In [72]:
def build_column_inventory(tables: dict, sample_n: int = 3, unique_threshold: int = 30) -> pd.DataFrame:
    rows = []

    for table_name, df in tables.items():
        row_count = df.shape[0]

        for col in df.columns:
            s = df[col]
            null_count = int(s.isna().sum())
            null_rate = (null_count / row_count) if row_count > 0 else 0.0
            n_unique = int(s.nunique(dropna=True))

            # 取前 sample_n 個非缺失樣本值
            sample_values = s.dropna().head(sample_n).tolist()

            # 若 n_unique 小於門檻，列出 unique 清單（排除缺失）
            unique_list = None
            if n_unique < unique_threshold:
                unique_list = s.dropna().unique().tolist()

            rows.append({
                "table_name": table_name,
                "column_name": col,
                "dtype": str(s.dtype),
                "row_count": row_count,
                "null_count": null_count,
                "null_rate": round(null_rate, 6),
                "n_unique": n_unique,
                "sample_values": sample_values,
                "unique_list": unique_list
            })

    column_inventory = (
        pd.DataFrame(rows)
        .sort_values(["table_name", "column_name"])
        .reset_index(drop=True)
    )
    return column_inventory

def add_business_meaning(
    column_inventory: pd.DataFrame,
    table_name: str,
    column_name: str,
    meaning: str
) -> pd.DataFrame:
    """
    在 column_inventory 中，為指定的 table + column 填入「欄位商業意義」
    若欄位不存在，會自動在第 3 欄（index=2）新增
    """

    # 若尚未有「欄位商業意義」欄位，先建立（放在第 3 欄）
    if "欄位商業意義" not in column_inventory.columns:
        column_inventory.insert(loc=2, column="欄位商業意義", value="")

    # 指定要填寫的 row
    mask = (
        (column_inventory["table_name"] == table_name) &
        (column_inventory["column_name"] == column_name)
    )

    # 寫入商業意義
    column_inventory.loc[mask, "欄位商業意義"] = meaning

    return column_inventory




In [157]:
column_inventory = build_column_inventory(
    tables=tables,
    sample_n=3,            # 每個欄位抓前 3 個非缺失樣本
    unique_threshold=30    # 唯一值少於 30 才列 unique_list
)

column_inventory.head(100)


,table_name,column_name,dtype,row_count,null_count,null_rate,n_unique,sample_values,unique_list
0,customer,customer_city,object,99441,0,0.000000,4119,"[franca, sao bernardo do campo, sao paulo]",None
1,customer,customer_id,object,99441,0,0.000000,99441,"[06b8999e2fba1a1fbc88172c00ba8bc7, 18955e83d337fd6b2def6b18a428ac77, 4e7b3e00288586ebd08712fdd0374a03]",None
2,customer,customer_state,object,99441,0,0.000000,27,"[SP, SP, SP]","[SP, SC, MG, PR, RJ, RS, PA, GO, ES, BA, MA, MS, CE, DF, RN, PE, MT, AM, AP, AL, RO, PB, TO, PI, AC, SE, RR]"
3,customer,customer_unique_id,object,99441,0,0.000000,96096,"[861eff4711a542e4b93843c6dd7febb0, 290c77bc529b7ac935b93aa66c333dc3, 060e732b5b29e8181a18229c7b0b2b5e]",None
4,customer,customer_zip_code_prefix,int64,99441,0,0.000000,14994,"[14409, 9790, 1151]",None
5,geolocation,geolocation_city,object,1000163,0,0.000000,8011,"[sao paulo, sao paulo, sao paulo]",None
6,geolocation,geolocation_lat,float64,1000163,0,0.000000,716685,"[-23.54562128, -23.54608113, -23.54612897]",None
7,geolocation,geolocation_lng,float64,1000163,0,0.000000,717097,"[-46.63929205, -46.6448203, -46.64295148]",None
8,geolocation,geolocation_state,object,1000163,0,0.000000,27,"[SP, SP, SP]","[SP, RN, AC, RJ, ES, MG, BA, SE, PE, AL, PB, CE, PI, MA, PA, AP, AM, RR, DF, GO, RO, TO, MT, MS, RS, PR, SC]"
9,geolocation,geolocation_zip_code_prefix,int64,1000163,0,0.000000,19015,"[1037, 1046, 1046]",None


### 使用excel進行欄位商業意義分析

In [86]:

target_dir = "/Users/kaiping/Desktop/olist_project/notebooks/data_understanding"
os.chdir(target_dir)
output_path = Path("column_inventory.csv")
if not output_path.exists():
    column_inventory.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"已儲存：{output_path}（位於 {os.getcwd()}）")
else:
    print(f"已存在，未覆蓋：{output_path}（位於 {os.getcwd()}）")

已存在，未覆蓋：column_inventory.csv（位於 /Users/kaiping/Desktop/olist_project/notebooks/data_understanding）


在進行欄位理解的時候，發現顧客表有兩個欄位 customer_id 和 customer_unique_id，我要驗證customer_id和  
order_id 是否數量一樣

訂單表 order_id (PK) 有99441筆
顧客表 customer_id(PK) 有99441筆
可以驗證只要客戶下單後就會產生一筆customer_id

In [ ]:
為什麼郵遞區號有四碼的還有五碼的？

In [ ]:
df_geolocation["geolocation_zip_code_prefix"].astype(str).str.len().value_counts()
# 讀取csv檔案時候開頭是0會被取消，巴西郵遞區號有0開頭的，後續資料處理要補0

訂單明細表的運費在同一筆訂單裡面理所應當要相同，需要驗證

In [97]:
check = (
    df_order_items
    .groupby("order_id")["freight_value"]
    .nunique()
    .reset_index(name="freight_value_cnt")
)

abnormal_orders = check[check["freight_value_cnt"] > 1]
abnormal_orders



,order_id,freight_value_cnt
73,002f98c0f7efd42638ed6100ca699b42,2
134,005d9a5423d47281ac463a968b3936fb,2
198,00946f674d880be1f188abc10ad7cf46,2
208,0097f0545a302aafa32782f1734ff71c,2
268,00bcee890eba57a9767c7b5ca12d3a1b,2
...,...,...
98555,ffb18bf111fa70edf316eb0390427986,2
98563,ffb8f7de8940249a3221252818937ecb,3
98564,ffb9a9cd00c74c11c24aa30b3d78e03b,3
98575,ffc16cecff8dc037f60458f28d1c1ba5,2


In [107]:
abnormal_orders["freight_value_cnt"].value_counts().sort_index()

# 訂單明細表中有兩種不同運費的有2000多筆
# 一筆訂單有大於1種不同的運費的資料，有4種，分別是 2,3,4,5

freight_value_cnt
2    1881
3     138
4      10
5       1
Name: count, dtype: int64

為何訂單明細表同一筆訂單會有多種不一樣的運費？  
推測是同一筆訂單買了多個seller的商品，每個seller的商品運費都不一樣，每個seller各自計算運費

In [108]:
# 每筆訂單：運費有幾種、賣家有幾個
order_level = (
    df_order_items
    .groupby("order_id")
    .agg(
        freight_cnt=("freight_value", "nunique"),
        seller_cnt=("seller_id", "nunique"),
        item_cnt=("order_item_id", "count")
    )
    .reset_index()
)

# 只看運費>1的訂單，有多少是 seller_cnt>1
pd.crosstab(order_level["freight_cnt"] > 1, order_level["seller_cnt"] > 1, margins=True)


seller_cnt,False,True,All
freight_cnt,,,
False,96393,243,96636
True,995,1035,2030
All,97388,1278,98666


In [113]:
# payment_installments 、 payment_sequential 代表啥意義？

df_order_payments.head(20)
# payment_installments 分期期數
# sequential 代表付款次序從1開始遞增


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45
5,298fcdf1f73eb413e4d26d01b25bc1cd,1,credit_card,2,96.12
6,771ee386b001f06208a7419e4fc1bbd7,1,credit_card,1,81.16
7,3d7239c394a212faae122962df514ac7,1,credit_card,3,51.84
8,1f78449c87a54faf9e96e88ba1491fa9,1,credit_card,6,341.09
9,0573b5e23cbd798006520e1d5b4c6714,1,boleto,1,51.95


In [121]:
print(f"付款次序:{np.sort(df_order_payments['payment_sequential'].unique())}\n")
print(f"分期期數:{np.sort(df_order_payments['payment_installments'].unique())}")

付款次序:[ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 26 27 28 29]

分期期數:[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 20 21 22 23 24]


為什麼付款次序最大29 ， 分期期數最大只有24?  
思考的結果是客人付款期數大於分期期數，超過預期期數  
或者是付款可能沒有成立

In [122]:
df_order_payments["payment_type"].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [128]:
print(f"共有{df_order_reviews['review_id'].count()}筆評論")

共有99224筆評論


In [146]:
print(df_orders["order_id"].nunique())
print(df_orders["customer_id"].nunique())

99441
99441


In [147]:
print(df_customers["customer_id"].nunique())

99441


訂單表的顧客id跟fk order_id數量相同，思考得出客人每下一筆訂單就會產生一個customer_id並且記錄客人個人資料

In [149]:
print(df_orders["order_status"].value_counts())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [156]:
df_products["product_photos_qty"].dropna().astype(int).sort_values().unique()


array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 17, 18,
       19, 20])

In [158]:
column_dictionary = pd.read_csv("column_inventory.csv")
column_dictionary


,table_name,column_name,business meaning,dtype,row_count,null_count,null_rate,n_unique,sample_values,unique_list
0,customer,customer_city,顧客所在地(市),object,99441,0,0.000000,4119,"['franca', 'sao bernardo do campo', 'sao paulo']",NaN
1,customer,customer_id,顧客下單快照(PK),object,99441,0,0.000000,99441,"['06b8999e2fba1a1fbc88172c00ba8bc7', '18955e83d337fd6b2def6b18a428ac77', '4e7b3e00288586ebd08712fdd0374a03']",NaN
2,customer,customer_state,顧客所在地(州)用兩碼代碼表示,object,99441,0,0.000000,27,"['SP', 'SP', 'SP']","['SP', 'SC', 'MG', 'PR', 'RJ', 'RS', 'PA', 'GO', 'ES', 'BA', 'MA', 'MS', 'CE', 'DF', 'RN', 'PE', 'MT', 'AM', 'AP', 'AL', 'RO', 'PB', 'TO', 'PI', 'AC', 'SE', 'RR']"
3,customer,customer_unique_id,平台推算的同一個真人，同一個人只能有一個代號,object,99441,0,0.000000,96096,"['861eff4711a542e4b93843c6dd7febb0', '290c77bc529b7ac935b93aa66c333dc3', '060e732b5b29e8181a18229c7b0b2b5e']",NaN
4,customer,customer_zip_code_prefix,巴西郵遞區號前五碼，通常代表很細的地理區域，有14000個唯一值，如果顯示四碼代表資料開頭是0 (excel讀int64欄位把0取消),int64,99441,0,0.000000,14994,"[14409, 9790, 1151]",NaN
5,geolocation,geolocation_city,巴西城市名稱有8000多唯一值,object,1000163,0,0.000000,8011,"['sao paulo', 'sao paulo', 'sao paulo']",NaN
6,geolocation,geolocation_lat,郵遞區號所對應的緯度(度),float64,1000163,0,0.000000,716685,"[-23.54562128, -23.54608113, -23.54612897]",NaN
7,geolocation,geolocation_lng,郵遞區號所對應的經度(度),float64,1000163,0,0.000000,717097,"[-46.63929205, -46.6448203, -46.64295148]",NaN
8,geolocation,geolocation_state,郵遞曲號所屬州 (兩碼代碼),object,1000163,0,0.000000,27,"['SP', 'SP', 'SP']","['SP', 'RN', 'AC', 'RJ', 'ES', 'MG', 'BA', 'SE', 'PE', 'AL', 'PB', 'CE', 'PI', 'MA', 'PA', 'AP', 'AM', 'RR', 'DF', 'GO', 'RO', 'TO', 'MT', 'MS', 'RS', 'PR', 'SC']"
9,geolocation,geolocation_zip_code_prefix,巴西郵遞區號前五碼，有19000唯一值，如果顯示四碼代表資料開頭是0 (excel讀int64欄位把0取消),int64,1000163,0,0.000000,19015,"[1037, 1046, 1046]",NaN
